# Stock Price Prediction with PyTorch

A walkthrough inspired by [NeuralNine’s tutorial](https://www.youtube.com/watch?v=IJ50ew8wi-0). This notebook uses the tested package in this repository. The default data is synthetic; install the package using the README before running all cells.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from stock_lstm.config import TrainConfig
from stock_lstm.data import prepare_data, read_prices, synthetic_prices
from stock_lstm.experiment import run_experiment
from stock_lstm.model import PriceLSTM
from stock_lstm.reporting import forecast_figure

%matplotlib inline

## 1. Choose the input

The seeded random walk keeps the walkthrough independent of a market-data provider. To use your downloaded prices, set `CSV_PATH` to their path and choose a label matching that file. Prices must have `Date` and `Close` columns.

In [ ]:
CSV_PATH = None  # Example: Path("../data/AAPL.csv")
config = TrainConfig(epochs=50, lookback=29, seed=42)
prices = read_prices(CSV_PATH) if CSV_PATH else synthetic_prices(seed=config.seed)
label = Path(CSV_PATH).stem if CSV_PATH else "SYNTHETIC random walk"
display(prices.head())
prices.set_index("Date").plot(title=label, figsize=(11, 3));

## 2. Respect time

Targets are split chronologically: 70% training, 10% validation, 20% test. A scaler is fitted only to training-period prices. Each input has 29 observations strictly before its target.

In [ ]:
prepared = prepare_data(prices, config)
display(
    pd.DataFrame(
        [
            {
                "split": name,
                "inputs": getattr(prepared, name).x.shape,
                "first_target": prepared.frame.Date.iloc[getattr(prepared, name).target_indices[0]],
                "last_target": prepared.frame.Date.iloc[getattr(prepared, name).target_indices[-1]],
            }
            for name in ("train", "validation", "test")
        ]
    )
)
print("Scaler fitted through:", prepared.frame.Date.iloc[prepared.scaler_fit_end - 1])

## 3. Inspect and train the network

Two LSTM layers with 32 hidden units feed a linear head. Adam minimizes MSE on standardized closes. The best validation checkpoint is restored before test evaluation.

In [ ]:
display(PriceLSTM(config.hidden_size, config.num_layers))
result = run_experiment(prices, config, label)
result.history.set_index("epoch").plot(title="Training and validation loss", figsize=(11, 3))
print("Selected epoch:", result.summary["best_epoch"])

## 4. Compare against persistence

Persistence predicts the last observed close. Both models are scored on identical dates in the original price units. The test chart uses earlier realized observations at each step; it is not a recursive forecast of the entire future period.

In [ ]:
display(pd.DataFrame(result.summary["metrics"]["test"]).T)
figure = forecast_figure(result)
display(figure)

## 5. Inspect the forecast and experiment record

This checkpoint was trained on the training period and selected using validation. It is not refitted to the final test observations. Synthetic results do not demonstrate market forecasting ability.

In [ ]:
display(result.summary["next_observation"])
print("Input SHA-256:", result.summary["data_sha256"])
print("Package versions:", result.summary["versions"])

## Next experiments

Use another ticker or time period, choose hyperparameters using validation only, and preserve a fresh final holdout. Record losses as well as wins against persistence. See `docs/methodology.md` for limitations and differences from the source tutorial.